In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import librosa
import json
import torch
import math
import soundfile as sf

In [2]:
audiocaps_wav_path = "/blob/v-yuancwang/TTADataset/processed_data/AudioCaps/wav"
audiocaps_info_path = "/blob/v-yuancwang/TTADataset/processed_data/AudioCaps/train_filter_speech.json"
with open(audiocaps_info_path, "r") as f:
    audiocaps_info = json.load(f)
traffic_info = []
children_info = []
beach_info = []
raining_info = []
traffic_key_words = ["traffic", "car", "bus", "motorcycle", "bicycle", "train"]
children_key_words = ["children", "child", "kid", "baby", "infant", "kids"]
beach_key_words = ["ocean wave", "ocean waving", "ocean waves", "ocean"]
raining_key_words = ["raining", "thundering", "storm", "thunder", "rain and thunder"]
for i in range(len(audiocaps_info)):
    if any([key_word in audiocaps_info[i]["Caption"] for key_word in traffic_key_words]):
        traffic_info.append(audiocaps_info[i])
    if any([key_word in audiocaps_info[i]["Caption"] for key_word in children_key_words]):
        children_info.append(audiocaps_info[i])
    if any([key_word in audiocaps_info[i]["Caption"] for key_word in beach_key_words]):
        beach_info.append(audiocaps_info[i])
    if any([key_word in audiocaps_info[i]["Caption"] for key_word in raining_key_words]):
        raining_info.append(audiocaps_info[i])
print("traffic_info:", len(traffic_info))
print("children_info:", len(children_info))
print("beach_info:", len(beach_info))
print("raining_info:", len(raining_info))
print(beach_info[0])

traffic_info: 2489
children_info: 855
beach_info: 41
raining_info: 335
{'Dataset': 'AudioCaps', 'Uid': '-CGRNKyzlzk_0_10000', 'Caption': 'Waves crashing on ocean shore'}


In [3]:
def get_noise_scale(wavform, noise, snr_min=5, snr_max=10):
    wavform_power = np.sum(wavform**2) / len(wavform)
    noise_power = np.sum(noise**2) / len(noise)
    snr = np.random.rand() * (snr_max - snr_min) + snr_min
    noise_scale = np.sqrt(wavform_power / (noise_power * 10 ** (snr / 10)))
    return noise_scale, snr


def calculate_snr(wavform, noise):
    wavform_power = np.sum(wavform**2) / len(wavform)
    noise_power = np.sum(noise**2) / len(noise)
    snr = 10 * math.log10(wavform_power / noise_power)
    return snr

def gen_env_speech(speech, env, snr_min, snr_max):
    noise_scale, snr = get_noise_scale(speech, env, snr_min, snr_max)
    return speech + env * noise_scale

In [4]:
speech_data_path = "/home/t-zeqianju/yuancwang/temp_speech_env/speech"
mix_data_path = "/home/t-zeqianju/yuancwang/temp_speech_env/mix"
test_json_path = "/home/t-zeqianju/yuancwang/Amphion/temp_meta_info/test_temp.json"

In [6]:
with open(test_json_path, "r") as f:
    test_info = json.load(f)
beach_sum = 0
traffic_sum = 0
children_sum = 0
raining_sum = 0
print(len(test_info))
for i in range(len(test_info)):
    if test_info[i]["background"] == "sea beach":
        beach_sum += 1
    if test_info[i]["background"] == "driving or traffic":
        traffic_sum += 1
    if test_info[i]["background"] == "children's voice":
        children_sum += 1
    if test_info[i]["background"] == "raining or thundering":
        raining_sum += 1
print("beach_sum:", beach_sum)
print("traffic_sum:", traffic_sum)
print("children_sum:", children_sum)
print("raining_sum:", raining_sum)

955
beach_sum: 240
traffic_sum: 245
children_sum: 235
raining_sum: 235


In [11]:
beach_sum = 0
traffic_sum = 0
children_sum = 0
raining_sum = 0
with open(test_json_path, "r") as f:
    test_info = json.load(f)
for i in range(len(test_info)):
    background = test_info[i]["background"]
    uid = test_info[i]["uid"]
    # choice env audio
    if background == "driving or traffic":
        env_info = traffic_info
        traffic_sum += 1
    elif background == "children's voice":
        env_info = children_info
        children_sum += 1
    elif background == "sea beach":
        env_info = beach_info
        beach_sum += 1
    elif background == "raining or thundering":
        env_info = raining_info
        raining_sum += 1
    random_env_uid = env_info[np.random.randint(0, len(env_info))]["Uid"]
    random_env_caption = env_info[np.random.randint(0, len(env_info))]["Caption"]
    random_env_path = os.path.join(audiocaps_wav_path, random_env_uid + ".wav")
    # load env audio
    env_audio = np.array([0.])
    while sum(env_audio) == 0:
        env_audio, sr = librosa.load(random_env_path, sr=16000)
        # load speech audio
        speech_path = os.path.join(speech_data_path, uid + ".wav")
        speech_audio, sr = librosa.load(speech_path, sr=16000)
        speech_len = len(speech_audio)
        env_len = len(env_audio)
        # padding env audio if env_len < speech_len
        if env_len < speech_len:
            env_audio = np.pad(env_audio, (0, speech_len - env_len), mode="wrap")
        if env_len > speech_len:
            env_audio_start = np.random.randint(0, env_len - speech_len)
        else:
            env_audio_start = 0
        env_audio = env_audio[env_audio_start:env_audio_start + speech_len]
    # print(len(speech_audio), len(env_audio))
    # mix audio
    mix_audio = gen_env_speech(speech_audio, env_audio, 5, 15)
    # save mix audio
    mix_path = os.path.join(mix_data_path, uid + ".wav")
    sf.write(mix_path, mix_audio, sr)

print("beach_sum:", beach_sum)
print("traffic_sum:", traffic_sum)
print("children_sum:", children_sum)
print("raining_sum:", raining_sum)

beach_sum: 240
traffic_sum: 245
children_sum: 235
raining_sum: 235
